# MIG Cement Demand Forecasting
### 02: Data Cleaning and Validation

### Purpose

The purpose of this notebook is to validate and clean the MIG cement operational data before exploratory data analysis and forecasting.

The cleaning process will ensure that the dataset is structurally consistent, internally valid, and suitable for subsequent analysis.

The following areas will be examined:

1. Duplicate records
2. Data types
3. Missing values
4. Invalid numerical values
5. Inventory balance consistency
6. Silo-capacity constraints
7. Site and cement-type references
8. Date continuity
9. Zero values
10. Extreme or unusual observations

 All cleaning operations will be performed on a separate working copy of the data.

### Loading necessary libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

import warnings
warnings.filterwarnings("ignore")


### loading the dataset 

In [2]:
data =pd.read_csv("../data/complete_dataset.csv")
data.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive


### Checking information about the dataset

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   cement_type               32880 non-null  str    
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
 11  region                    32880 non-null  str    
 12  behavior                  32880 non-null  str    
dtypes: float64(7), int64(1), str(5)
memory usage: 3.3 MB


-  Data issue spotted at glance

- date column --> to datetime

In [4]:
# Dealing with the datetime column
data['date'] = pd.to_datetime(data['date'])

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      32880 non-null  datetime64[us]
 1   site_id                   32880 non-null  str           
 2   cement_type               32880 non-null  str           
 3   planned_pour_tonnes       32880 non-null  float64       
 4   consumed_tonnes           32880 non-null  float64       
 5   opening_inventory_tonnes  32880 non-null  float64       
 6   deliveries_tonnes         32880 non-null  float64       
 7   closing_inventory_tonnes  32880 non-null  float64       
 8   rain_mm                   32880 non-null  float64       
 9   avg_temp_c                32880 non-null  float64       
 10  silo_capacity             32880 non-null  int64         
 11  region                    32880 non-null  str           
 12  behavior                  328

### Checking for missing values

In [6]:
data.isnull().sum()

date                        0
site_id                     0
cement_type                 0
planned_pour_tonnes         0
consumed_tonnes             0
opening_inventory_tonnes    0
deliveries_tonnes           0
closing_inventory_tonnes    0
rain_mm                     0
avg_temp_c                  0
silo_capacity               0
region                      0
behavior                    0
dtype: int64

- It seems there is not missing values but we are going to gig further to see if there any missing value issue here

#### Duplicate values

In [7]:
data.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
32875    False
32876    False
32877    False
32878    False
32879    False
Length: 32880, dtype: bool

In [8]:
data.duplicated().sum()

np.int64(0)

-  No duplicated values as well

In [9]:
data.duplicated(
    subset=["date", "site_id", "cement_type"]
).sum()

np.int64(0)

#### Check Invalid Numerical values

In [10]:
invalid_values = {
    "Negative planned pours": (data["planned_pour_tonnes"] < 0).sum(),
    "Negative consumption": (data["consumed_tonnes"] < 0).sum(),
    "Negative opening inventory": (data["opening_inventory_tonnes"] < 0).sum(),
    "Negative deliveries": (data["deliveries_tonnes"] < 0).sum(),
    "Negative closing inventory": (data["closing_inventory_tonnes"] < 0).sum(),
    "Negative rainfall": (data["rain_mm"] < 0).sum(),
    "Invalid silo capacity": (data["silo_capacity"] <= 0).sum()
}

pd.Series(invalid_values)

Negative planned pours        0
Negative consumption          0
Negative opening inventory    0
Negative deliveries           0
Negative closing inventory    0
Negative rainfall             0
Invalid silo capacity         0
dtype: int64

- We checked this because we expected all these features to have postive values (>=0) aside teampeature that could have negative values.

- This look good. Let's move

### Data Quality Checks (Consistency check)

Inventory records should satisfy the following operational relationship:

**Closing Inventory = Opening Inventory + Deliveries - Consumption**

This relationship is used as an internal consistency check.

For each record, an expected closing inventory is calculated from the opening inventory, deliveries, and actual cement consumption. The calculated value is then compared with the recorded closing inventory.

Because the dataset contains decimal quantities, values are rounded to two decimal places before comparison to avoid treating insignificant floating-point differences as data-quality errors.

Records that fail this balance check will be investigated before any correction is made.

In [11]:
expected_closing = (
    data["opening_inventory_tonnes"]
    + data["deliveries_tonnes"]
    - data["consumed_tonnes"]
).round(2)

In [12]:
expected_closing

0        63.85
1        38.56
2        47.06
3        32.64
4         0.00
         ...  
32875     0.00
32876     0.00
32877     0.00
32878     0.00
32879     0.00
Length: 32880, dtype: float64

In [13]:
data['closing_inventory_tonnes']

0        63.85
1        38.56
2        47.06
3        32.64
4         0.00
         ...  
32875     0.00
32876     0.00
32877     0.00
32878     0.00
32879     0.00
Name: closing_inventory_tonnes, Length: 32880, dtype: float64

In [14]:
balance_difference = (
    data["closing_inventory_tonnes"]
    - expected_closing
).round(2)

In [15]:
balance_issues_count = (balance_difference != 0).sum()

print("Inventory balance issues:", balance_issues_count)

Inventory balance issues: 7


In [16]:
balance_issues = data.loc[
    balance_difference != 0,
    [
        "date",
        "site_id",
        "cement_type",
        "opening_inventory_tonnes",
        "deliveries_tonnes",
        "consumed_tonnes",
        "closing_inventory_tonnes"
    ]
].copy()

balance_issues["expected_closing_inventory"] = expected_closing[
    balance_difference != 0
]

balance_issues["difference"] = balance_difference[
    balance_difference != 0
]

balance_issues

,date,site_id,cement_type,opening_inventory_tonnes,deliveries_tonnes,consumed_tonnes,closing_inventory_tonnes,expected_closing_inventory,difference
1096,2022-01-01,SITE_002,CEM_I,28.46,12.12,15.70,24.87,24.88,-0.01
13152,2022-01-01,SITE_013,CEM_III,24.44,25.63,50.08,0.00,-0.01,0.01
15344,2022-01-01,SITE_015,CEM_I,26.95,18.36,9.46,35.86,35.85,0.01
25208,2022-01-01,SITE_024,CEM_II,73.23,36.61,64.29,45.56,45.55,0.01
28496,2022-01-01,SITE_027,CEM_II,62.79,13.10,12.38,63.52,63.51,0.01
30688,2022-01-01,SITE_029,CEM_II,53.68,39.26,16.39,76.54,76.55,-0.01
31784,2022-01-01,SITE_030,CEM_II,66.26,24.24,39.27,51.22,51.23,-0.01


-  This is not a serious issues, I assume it is approximation issues, since we do not have access to the full approximation figures, we cannot compare them at that same precision. So I won't treat it as a problem

#### Validate Silo Capacity

The `silo_capacity` variable represents the maximum quantity of cement that can be stored at a construction site.

Inventory values should therefore normally satisfy the following conditions:

- Opening Inventory ≤ Silo Capacity
- Closing Inventory ≤ Silo Capacity

Any record exceeding the stated silo capacity may indicate a data-quality issue or an operational situation that requires further investigation.

In addition, silo capacity is stored in both the `Operations` and `Sites` tables. The values will therefore be checked for consistency across the two tables.

In [17]:
opening_capacity_issues = (
    data["opening_inventory_tonnes"]
    > data["silo_capacity"]
).sum()

print("Opening inventory above silo capacity:", opening_capacity_issues)

Opening inventory above silo capacity: 11427


In [18]:
data[data["opening_inventory_tonnes"] > data["silo_capacity"]]

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
1110,2022-01-15,SITE_002,CEM_I,11.11,11.11,292.12,19.07,300.08,2.06,13.29,288,South,conservative
1111,2022-01-16,SITE_002,CEM_II,18.42,18.42,300.08,26.42,308.08,4.61,16.74,288,South,conservative
1112,2022-01-17,SITE_002,CEM_II,11.46,11.46,308.08,27.52,324.14,11.44,14.27,288,South,conservative
1113,2022-01-18,SITE_002,CEM_II,8.59,8.59,324.14,20.05,335.60,2.63,16.19,288,South,conservative
1114,2022-01-19,SITE_002,CEM_II,10.39,10.39,335.60,25.52,350.73,1.93,10.99,288,South,conservative
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31779,2024-12-27,SITE_029,CEM_I,7.64,6.11,20396.88,45.44,20436.21,0.39,-2.37,437,West,conservative
31780,2024-12-28,SITE_029,CEM_II,10.06,10.06,20436.21,11.90,20438.05,0.13,0.12,437,West,conservative
31781,2024-12-29,SITE_029,CEM_III,7.92,7.92,20438.05,26.40,20456.53,8.43,9.01,437,West,conservative
31782,2024-12-30,SITE_029,CEM_II,15.50,15.50,20456.53,15.09,20456.12,3.75,3.36,437,West,conservative


In [19]:
closing_capacity_issues = (
    data["closing_inventory_tonnes"]
    > data["silo_capacity"]
).sum()

print("Closing inventory above silo capacity:", closing_capacity_issues)

Closing inventory above silo capacity: 11439


In [20]:
data[data["closing_inventory_tonnes"] > data["silo_capacity"]]

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
1109,2022-01-14,SITE_002,CEM_III,14.99,14.99,271.54,35.57,292.12,8.51,9.85,288,South,conservative
1110,2022-01-15,SITE_002,CEM_I,11.11,11.11,292.12,19.07,300.08,2.06,13.29,288,South,conservative
1111,2022-01-16,SITE_002,CEM_II,18.42,18.42,300.08,26.42,308.08,4.61,16.74,288,South,conservative
1112,2022-01-17,SITE_002,CEM_II,11.46,11.46,308.08,27.52,324.14,11.44,14.27,288,South,conservative
1113,2022-01-18,SITE_002,CEM_II,8.59,8.59,324.14,20.05,335.60,2.63,16.19,288,South,conservative
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31779,2024-12-27,SITE_029,CEM_I,7.64,6.11,20396.88,45.44,20436.21,0.39,-2.37,437,West,conservative
31780,2024-12-28,SITE_029,CEM_II,10.06,10.06,20436.21,11.90,20438.05,0.13,0.12,437,West,conservative
31781,2024-12-29,SITE_029,CEM_III,7.92,7.92,20438.05,26.40,20456.53,8.43,9.01,437,West,conservative
31782,2024-12-30,SITE_029,CEM_II,15.50,15.50,20456.53,15.09,20456.12,3.75,3.36,437,West,conservative


In [21]:
pd.set_option("display.max_rows", None)

In [22]:
data.loc[
    data["opening_inventory_tonnes"] > data["silo_capacity"],
    [
        "date",
        "site_id",
        "cement_type",
        "opening_inventory_tonnes",
        "silo_capacity"
    ]
].tail(10)

,date,site_id,cement_type,opening_inventory_tonnes,silo_capacity
31774,2024-12-22,SITE_029,CEM_III,20308.19,437
31775,2024-12-23,SITE_029,CEM_I,20313.59,437
31776,2024-12-24,SITE_029,CEM_II,20312.12,437
31777,2024-12-25,SITE_029,CEM_I,20347.31,437
31778,2024-12-26,SITE_029,CEM_III,20365.70,437
31779,2024-12-27,SITE_029,CEM_I,20396.88,437
31780,2024-12-28,SITE_029,CEM_II,20436.21,437
31781,2024-12-29,SITE_029,CEM_III,20438.05,437
31782,2024-12-30,SITE_029,CEM_II,20456.53,437
31783,2024-12-31,SITE_029,CEM_III,20456.12,437


In [23]:
data.loc[
    data["closing_inventory_tonnes"] > data["silo_capacity"],
    [
        "date",
        "site_id",
        "cement_type",
        "closing_inventory_tonnes",
        "silo_capacity"
    ]
].tail(10)

,date,site_id,cement_type,closing_inventory_tonnes,silo_capacity
31774,2024-12-22,SITE_029,CEM_III,20313.59,437
31775,2024-12-23,SITE_029,CEM_I,20312.12,437
31776,2024-12-24,SITE_029,CEM_II,20347.31,437
31777,2024-12-25,SITE_029,CEM_I,20365.70,437
31778,2024-12-26,SITE_029,CEM_III,20396.88,437
31779,2024-12-27,SITE_029,CEM_I,20436.21,437
31780,2024-12-28,SITE_029,CEM_II,20438.05,437
31781,2024-12-29,SITE_029,CEM_III,20456.53,437
31782,2024-12-30,SITE_029,CEM_II,20456.12,437
31783,2024-12-31,SITE_029,CEM_III,20472.10,437


In [24]:
# Which stites have the most opening inventory  more than silo capacity
opening_violations_by_site = (
    data.loc[
        data["opening_inventory_tonnes"] > data["silo_capacity"]
    ]
    ["site_id"]
    .value_counts()
)

opening_violations_by_site

site_id
SITE_015    1090
SITE_012    1085
SITE_023    1084
SITE_002    1082
SITE_009    1082
SITE_027    1081
SITE_029    1076
SITE_004    1074
SITE_019    1074
SITE_013     557
SITE_024     366
SITE_014     352
SITE_028     183
SITE_016     168
SITE_026      73
Name: count, dtype: int64

In [25]:
closing_violations_by_site = (
    data.loc[
        data["closing_inventory_tonnes"] > data["silo_capacity"]
    ]
    ["site_id"]
    .value_counts()
)

closing_violations_by_site

site_id
SITE_015    1091
SITE_012    1086
SITE_023    1085
SITE_002    1083
SITE_009    1083
SITE_027    1082
SITE_029    1077
SITE_004    1075
SITE_019    1075
SITE_013     557
SITE_024     367
SITE_014     353
SITE_028     184
SITE_016     168
SITE_026      73
Name: count, dtype: int64

### Findings

The silo-capacity validation identified a substantial number of records where recorded inventory exceeded the stated site silo capacity.

- 11,427 records had opening inventory above silo capacity.
- 11,439 records had closing inventory above silo capacity.

Further inspection showed that the violations were systematic rather than randomly distributed. They were heavily concentrated among sites classified as `conservative`, while `aggressive` sites did not exhibit capacity violations and `chaotic` sites showed violations less frequently.

The inventory balances also remain sequentially consistent, with one day's closing inventory carrying forward as the following day's opening inventory. This suggests that the excessive inventory values arise from cumulative operational behavior in the dataset rather than isolated recording errors.

Because overstocking is itself an important business problem in this project, these observations will not be deleted or capped during data cleaning. Instead, capacity-violation indicators will be retained so that the pattern can be investigated during exploratory analysis and incorporated into later inventory-risk assessment.

The physical interpretation of inventory levels greatly exceeding stated silo capacity should nevertheless be treated as a dataset limitation and validated before any real-world deployment.

In [26]:
# Let's flag these issues in the original dataset for further analysis
data["opening_capacity_violation"] = (
    data["opening_inventory_tonnes"] > data["silo_capacity"]
)

data["closing_capacity_violation"] = (
    data["closing_inventory_tonnes"] > data["silo_capacity"]
)

### Checking whether Silo Capacity changes with Site

In [27]:
capacity_per_site = (
    data.groupby("site_id")["silo_capacity"]
    .nunique()
)

capacity_per_site.tail

<bound method NDFrame.tail of site_id
SITE_001    1
SITE_002    1
SITE_003    1
SITE_004    1
SITE_005    1
SITE_006    1
SITE_007    1
SITE_008    1
SITE_009    1
SITE_010    1
SITE_011    1
SITE_012    1
SITE_013    1
SITE_014    1
SITE_015    1
SITE_016    1
SITE_017    1
SITE_018    1
SITE_019    1
SITE_020    1
SITE_021    1
SITE_022    1
SITE_023    1
SITE_024    1
SITE_025    1
SITE_026    1
SITE_027    1
SITE_028    1
SITE_029    1
SITE_030    1
Name: silo_capacity, dtype: int64>

### Data Check

SInce this is a forecasting project, we need to know whether each site's historical record runs continuously through time or whether some dates are missing.

In [28]:
date_continuity = (
    data.groupby("site_id")
        .agg(
            start_date=("date", "min"),
            end_date=("date", "max"),
            observed_days=("date", "nunique")
        )
)

date_continuity["expected_days"] = (
    date_continuity["end_date"] -
    date_continuity["start_date"]
).dt.days + 1

date_continuity["missing_days"] = (
    date_continuity["expected_days"]
    - date_continuity["observed_days"]
)

date_continuity

,start_date,end_date,observed_days,expected_days,missing_days
site_id,,,,,
SITE_001,2022-01-01,2024-12-31,1096,1096,0
SITE_002,2022-01-01,2024-12-31,1096,1096,0
SITE_003,2022-01-01,2024-12-31,1096,1096,0
SITE_004,2022-01-01,2024-12-31,1096,1096,0
SITE_005,2022-01-01,2024-12-31,1096,1096,0
SITE_006,2022-01-01,2024-12-31,1096,1096,0
SITE_007,2022-01-01,2024-12-31,1096,1096,0
SITE_008,2022-01-01,2024-12-31,1096,1096,0
SITE_009,2022-01-01,2024-12-31,1096,1096,0


- It very clear there is no gap in dates

### Investigate Zero Values

In [29]:
zero_counts = {
    "planned_pour_tonnes": (data["planned_pour_tonnes"] == 0).sum(),
    "consumed_tonnes": (data["consumed_tonnes"] == 0).sum(),
    "opening_inventory_tonnes": (data["opening_inventory_tonnes"] == 0).sum(),
    "deliveries_tonnes": (data["deliveries_tonnes"] == 0).sum(),
    "closing_inventory_tonnes": (data["closing_inventory_tonnes"] == 0).sum(),
    "rain_mm": (data["rain_mm"] == 0).sum()
}

pd.Series(zero_counts)

planned_pour_tonnes         2993
consumed_tonnes             4003
opening_inventory_tonnes    7942
deliveries_tonnes            808
closing_inventory_tonnes    7950
rain_mm                       26
dtype: int64

In [30]:
data[data['planned_pour_tonnes']==0].head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior,opening_capacity_violation,closing_capacity_violation
8,2022-01-09,SITE_001,CEM_I,0.0,0.0,0.00,34.38,34.38,1.42,16.81,448,North,aggressive,False,False
43,2022-02-13,SITE_001,CEM_II,0.0,0.0,0.00,47.34,47.34,14.89,11.62,448,North,aggressive,False,False
91,2022-04-02,SITE_001,CEM_II,0.0,0.0,7.43,44.69,52.12,11.28,21.26,448,North,aggressive,False,False
92,2022-04-03,SITE_001,CEM_III,0.0,0.0,52.12,22.88,75.00,2.19,15.64,448,North,aggressive,False,False
126,2022-05-07,SITE_001,CEM_III,0.0,0.0,0.00,36.54,36.54,0.17,23.46,448,North,aggressive,False,False


In [31]:
zero_closing_by_site = (
    data.loc[data["closing_inventory_tonnes"] == 0]
        ["site_id"]
        .value_counts()
)

zero_closing_by_site

site_id
SITE_022    563
SITE_008    547
SITE_011    546
SITE_021    535
SITE_020    531
SITE_017    525
SITE_010    518
SITE_005    516
SITE_025    515
SITE_001    510
SITE_007    507
SITE_030    503
SITE_018    498
SITE_003    497
SITE_006    147
SITE_016    110
SITE_028     90
SITE_013     89
SITE_026     85
SITE_014     67
SITE_024     51
Name: count, dtype: int64

### Checking for Extreme numerical observations/Outliers

In [32]:
numerical_cols = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity"
]

data[numerical_cols].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
planned_pour_tonnes,32880.0,30.697158,19.493713,0.0,0.00,12.83,33.455,47.6125,66.4342,69.98
consumed_tonnes,32880.0,23.720475,16.846251,0.0,0.00,10.71,19.720,36.4925,64.4884,69.97
opening_inventory_tonnes,32880.0,3066.682205,5586.029168,0.0,0.00,1.28,50.710,3298.5225,19608.0095,20646.18
deliveries_tonnes,32880.0,29.293009,12.325600,0.0,0.00,19.30,29.490,39.7500,49.6121,50.00
closing_inventory_tonnes,32880.0,3072.254739,5593.015910,0.0,0.00,1.27,50.660,3312.0625,19631.6346,20658.87
rain_mm,32880.0,5.008120,4.997142,0.0,0.05,1.44,3.470,6.9700,23.3000,50.00
avg_temp_c,32880.0,10.085887,8.519946,-5.0,-5.00,3.34,9.990,16.7000,28.2421,35.00
silo_capacity,32880.0,317.533333,112.813279,120.0,120.00,230.00,314.000,437.0000,487.0000,487.00


In [33]:
outlier_summary = []

for col in numerical_cols:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (data[col] < lower_bound) |
        (data[col] > upper_bound)
    ).sum()

    outlier_summary.append({
        "variable": col,
        "lower_bound": round(lower_bound, 2),
        "upper_bound": round(upper_bound, 2),
        "potential_outliers": outliers
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,variable,lower_bound,upper_bound,potential_outliers
0,planned_pour_tonnes,-39.34,99.79,0
1,consumed_tonnes,-27.96,75.17,0
2,opening_inventory_tonnes,-4944.58,8244.39,5799
3,deliveries_tonnes,-11.37,70.42,0
4,closing_inventory_tonnes,-4964.92,8278.25,5793
5,rain_mm,-6.85,15.26,1552
6,avg_temp_c,-16.70,36.74,0
7,silo_capacity,-80.50,747.50,0


### Data Cleaning  Summary

In [34]:
quality_summary = {
    "Total rows": len(data),
    "Total columns": data.shape[1],
    "Exact duplicates": data.duplicated().sum(),
    "Natural-key duplicates": data.duplicated(
        subset=["date", "site_id", "cement_type"]
    ).sum(),
    "Missing values": data.isna().sum().sum(),
    "Negative consumption": (data["consumed_tonnes"] < 0).sum(),
    "Negative deliveries": (data["deliveries_tonnes"] < 0).sum(),
    "Inventory balance issues": (
        (
            data["closing_inventory_tonnes"]
            - (
                data["opening_inventory_tonnes"]
                + data["deliveries_tonnes"]
                - data["consumed_tonnes"]
            )
        ).round(2) != 0
    ).sum(),
    "Opening capacity violations": (
        data["opening_inventory_tonnes"] > data["silo_capacity"]
    ).sum(),
    "Closing capacity violations": (
        data["closing_inventory_tonnes"] > data["silo_capacity"]
    ).sum()
}

pd.Series(quality_summary)

Total rows                     32880
Total columns                     15
Exact duplicates                   0
Natural-key duplicates             0
Missing values                     0
Negative consumption               0
Negative deliveries                0
Inventory balance issues           7
Opening capacity violations    11427
Closing capacity violations    11439
dtype: int64

### Final Data Quality Summary

The MIG cement operational dataset was systematically validated before exploratory analysis and forecasting.

The cleaning process examined duplicate records, data types, missing values, invalid numerical observations, inventory accounting consistency, silo-capacity constraints, reference-table integrity, chronological continuity, zero values, and extreme observations.

### Data-Type Validation

The `date` variable was converted from text to a proper datetime format to support chronological analysis and subsequent forecasting.

The remaining numerical and categorical variables were retained in appropriate formats.

### Duplicate Records

Exact duplicate records and duplicate combinations of `date`, `site_id`, and `cement_type` were checked.

The `(date, site_id, cement_type)` combination was treated as the natural operational record key.

### Missing Values

Missing values were assessed across the operational and reference tables.

No missing values were automatically imputed. Any missing information identified during validation was to be investigated according to its business meaning rather than filled using a generic rule.

### Invalid Numerical Values

Core numerical variables were checked for logically impossible values, including negative cement consumption, negative deliveries, negative inventory, negative planned pours, negative rainfall, and non-positive silo capacity.

Extreme but potentially legitimate values were treated separately from logically invalid observations.

### Inventory Balance Validation

Each operational record was checked against the inventory relationship:

**Closing Inventory = Opening Inventory + Deliveries - Consumption**

This provided an internal accounting consistency test for the inventory records.

### Silo-Capacity Validation

A substantial number of inventory observations exceeded the site's stated silo capacity:

- **11,427 opening-inventory observations exceeded silo capacity**
- **11,439 closing-inventory observations exceeded silo capacity**

Further investigation showed that these violations were systematic rather than isolated.

The violations were strongly associated with site operating patterns, particularly sites classified as `conservative`, while aggressive sites exhibited substantially fewer or no such violations.

Sequential inventory records were also observed to carry closing inventory correctly into subsequent opening inventory values. This indicates that the excessive inventory levels arise from cumulative inventory behavior rather than isolated arithmetic errors.

Because overstocking is itself part of the business problem being investigated, these observations were **not deleted or artificially capped**.

Instead, two indicators were created:

- `opening_capacity_violation`
- `closing_capacity_violation`

These variables preserve the original information while allowing over-capacity inventory conditions to be analyzed explicitly.

The physical interpretation of inventory values that substantially exceed stated silo capacity remains an important dataset limitation and would require operational validation before real-world deployment.

### Site and Cement-Type Validation

Site identifiers were checked against the `Sites` reference table and cement-type identifiers were checked against the `CementTypes` reference table.

The `region` and `behavior` fields were also inspected as site-level categorical information.

The `behavior` variable was retained as supplied by the database. Its categories include `aggressive`, `conservative`, and `chaotic`. Because the source documentation does not define how these labels were assigned, their exact operational interpretation will be investigated during exploratory analysis before considering predictive use.

### Date Continuity

Site-level histories were examined for gaps between each site's first and last observation.

Missing calendar dates were not automatically converted into zero-demand observations because a missing day and a genuine zero-consumption day have different business meanings.

### Zero Values

Zero values were retained.

A zero can represent a legitimate operational event, including:

- no scheduled pour,
- no cement consumption,
- no delivery,
- no rainfall, or
- zero remaining inventory.

In particular, zero closing inventory may later provide important information about stockout risk and pour readiness.

### Extreme Observations

Extreme numerical observations were investigated rather than automatically removed.

Construction demand, deliveries, and inventory can naturally be volatile, and unusually large observations may represent genuine operational events.

The IQR method was therefore used as an investigation tool rather than an automatic deletion rule.

### Cleaning Principle

The cleaning process followed a conservative approach:

> **Correct genuine structural problems, flag important operational anomalies, and preserve valid business variation.**

No observation was removed or modified solely because it was unusual.

### Cleaning Actions Applied

The following transformations were applied during the cleaning stage:

1. Converted `date` to datetime format.
2. Preserved the original raw database without modification.
3. Retained valid zero observations.
4. Retained extreme but operationally plausible observations.
5. Retained inventory values exceeding silo capacity rather than artificially capping them.
6. Added `opening_capacity_violation` to identify opening inventory above stated capacity.
7. Added `closing_capacity_violation` to identify closing inventory above stated capacity.
8. Preserved site-level `region` and `behavior` information for subsequent exploratory analysis.

No unnecessary imputation, outlier deletion, or arbitrary value replacement was performed.

In [35]:
data.to_csv("../data/cleaned_dataset.csv", index=False)